# Generative models on aerial roof images: hands-on lab

**Guest lecture · AAI6640 Module 13 · The Roux Institute at Northeastern University · November 19, 2026**
Dr. Sushanth Manakhari · hosted by Dr. Umesh Hodeghatta Rao

You have just joined the property team of a fictional home insurer. Four tickets are waiting, one for each part
of Module 13, and a fifth part shows what today's open models do better:

| Part | Ticket | Model | Time |
|---|---|---|---|
| 1 | **ROOF-301** | Autoencoder: find unusual roofs worth a second look | about 17 min |
| 2 | **ROOF-302** | Variational autoencoder (VAE): a smooth "roof space" to sample and morph | about 13 min |
| 3 | **ROOF-303** | GAN: a generator and a critic compete until fakes look real | about 18 min |
| 4 | Modern fixes | Why our pictures blur, and two open models that sharpen them | about 10 min |
| 5 | **ROOF-304** | Ethics: copying, one score per home, fairness, fake evidence | about 10 min |

**How to use this notebook**
- Menu **Runtime → Change runtime type → T4 GPU → Save** first.
- Run the cells in order, top to bottom: click a cell, then press **Shift + Enter**.
- Cells marked **Exercise** are yours. Cells marked **Instructor demo** are shown on the screen; you can run them
  afterwards.
- Every roof comes from free, public aerial photographs. The packs hold **no addresses, coordinates or owner
  details**. Sources and licences: [docs/DATA_SOURCES.md](https://github.com/Sushanth-Manakhari/generative-models-aerial-imagery/blob/main/docs/DATA_SOURCES.md).

In [ ]:
# Setup 1 of 2: install the course package (a few seconds)
%pip install -q "genroofs[modern] @ git+https://github.com/Sushanth-Manakhari/generative-models-aerial-imagery.git@main"

In [ ]:
# Setup 2 of 2: the tools, the device and the roofs (about 45 seconds the first time)
import torch
import torch.nn.functional as F
from genroofs import data, train, metrics, viz, exercises, pipeline, modern
from genroofs.models import RoofAE, RoofVAE, Generator, Discriminator
from genroofs.utils import pick_device, set_seed

DEVICE = pick_device()
print("Running on:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else DEVICE.type)

# True = load the ready-made models instead of training them (no GPU, or short on time). They use the same
# designs and recipes, trained before class on the Maine roofs, leaving out the class test roofs.
READY_MADE = DEVICE.type != "cuda"
if READY_MADE:
    print("No GPU here, so this notebook loads ready-made models instead of training them.\n"
          "To get a GPU: Runtime -> Change runtime type -> T4 GPU -> Save, then run this cell again.")

roofs = data.load_roofs("roofs_maine_2022")               # 22,000 Maine roofs at 64 x 64 pixels
tr, te = data.train_test_split(data.class_slice(roofs))   # the class's 10,000: 9,000 to train, 1,000 to test
variety = data.load_roofs("roofs_us_variety")             # 3,000 roofs from five other states
print(f"training roofs: {len(tr.images):,} · test roofs: {len(te.images):,} · variety roofs: {len(variety.images):,}")
viz.show_grid(te.images[:16], ncol=8, title="16 of the class test roofs (each square is 24 m across)")

Each roof is a **64 × 64 colour picture**: 64 × 64 pixels × 3 colours (red, green, blue) = **12,288 numbers**.
The models see each number on a scale from −1 (dark) to +1 (bright).

---
# Part 1 · ROOF-301 · Autoencoder

> **Ticket ROOF-301.** *Inspectors can't visit every home. Find the roofs that look unusual, so they get a second
> look first.*

**The idea.** An autoencoder has two halves:
- the **encoder** squeezes a roof's 12,288 numbers into just **128 numbers** (the *bottleneck*, or *latent code*);
- the **decoder** rebuilds a 64 × 64 roof from those 128 numbers.

It is trained to make the copy as close to the original as possible. Because 128 numbers can't hold everything,
it learns what typical Maine roofs have in common. A roof it copies **badly** is one that doesn't fit the
pattern: that is our "unusual" score.

```
roof (12,288 numbers) ──encoder──▶ 128 numbers ──decoder──▶ copy (12,288 numbers)
```

**The loss: how wrong the copies are**

$$L \;=\; \frac{1}{B}\sum_{i=1}^{B} \big\lVert x_i - \hat{x}_i \big\rVert^2$$

| Symbol | Meaning |
|---|---|
| $x_i$ | roof number $i$ in the batch: its 12,288 pixel values between −1 and 1 |
| $\hat{x}_i$ | the decoder's copy of that roof ("x hat") |
| $x_i - \hat{x}_i$ | the error at every pixel |
| $\lVert\cdot\rVert^2$ | square each pixel's error and add them all up (squaring makes every error positive and punishes big ones most) |
| $B$ | the batch size: 128 roofs per training step |
| $\frac{1}{B}\sum_{i=1}^{B}$ | the average over the roofs in the batch |

**Worked example.** A tiny "roof" of 3 pixels, $x = (0.2,\ -0.5,\ 1.0)$, and its copy $\hat{x} = (0.1,\ -0.5,\ 0.6)$.
The errors are $(0.1,\ 0,\ 0.4)$, their squares $(0.01,\ 0,\ 0.16)$, and their sum $0.17$. If a second roof in a
batch of $B = 2$ has squared error $0.05$, then $L = (0.17 + 0.05) / 2 = 0.11$. Training nudges the network's
weights, step after step, to make $L$ smaller.

In [ ]:
# Train the autoencoder for 1 minute (about 80 passes over the 9,000 training roofs on a T4)
set_seed(7)
if READY_MADE:
    ae = train.load_pretrained("ae")
else:
    ae = RoofAE()
    h_ae = train.train_ae(ae, tr.images, watch=te.images[:8])
    viz.show_snapshots(h_ae)      # the same 8 test roofs, copied as training went on
    viz.plot_history(h_ae)        # the loss falling

**How close are the copies?** We measure it with PSNR, the *peak signal-to-noise ratio*, in decibels (dB):

$$\text{PSNR} = 10 \cdot \log_{10}\!\left(\frac{R^2}{\text{MSE}}\right)$$

| Symbol | Meaning |
|---|---|
| MSE | the *mean* squared error per pixel (the loss above, divided by 12,288) |
| $R$ | the range of pixel values: from −1 to 1, so $R = 2$ and $R^2 = 4$ |
| $\log_{10}$ | how many times you multiply 10 to get the number (so each +10 dB means 10 times less error) |

**Worked example.** If the mean squared error is $0.02$: $4 / 0.02 = 200$, $\log_{10} 200 \approx 2.3$, so
PSNR $\approx 23$ dB. About 20 dB looks blurry; about 30 dB is hard to tell apart from the original.

In [ ]:
copies = exercises.copies(ae, te.images[:8])
viz.show_pairs(te.images[:8], copies, title="Test roofs (top) and the autoencoder's copies (bottom)")
print(f"Average PSNR on the 1,000 test roofs: {metrics.psnr(te.images, exercises.copies(ae, te.images)).mean():.1f} dB")

### Exercise 1 · How big should the bottleneck be? (2 minutes)

Our autoencoder keeps **128 numbers** per roof. The next cell trains a second one, for 1 minute, that keeps only
**8**. Before you run it, guess: will its copies be a little worse, or much worse?

Then look at the pictures and the two PSNR values. *What kind of detail disappears first?*

In [ ]:
# Exercise 1: an autoencoder with only 8 numbers in its bottleneck (1 minute)
r1 = exercises.bottleneck(ae, tr.images, te.images, latent=8, minutes=1.0)

**The unusual score.** For every roof, the autoencoder's total squared error $\lVert x - \hat{x}\rVert^2$ is its
*unusual score*: high means "this roof doesn't look like the Maine roofs I learned from".

Is that score any good? We test it on roofs we *know* are different: 3,000 roofs from Connecticut, Indiana, Texas,
Florida and Arizona that the model never saw. The **AUC** answers one question: *pick one of those roofs and one
Maine test roof at random; how often does the other-state roof get the higher score?*

| AUC | Meaning |
|---|---|
| 1.0 | always: a perfect detector |
| 0.5 | half the time: a coin toss |
| below 0.5 | worse than a coin toss |

**Worked example.** Three Maine roofs score 10, 20 and 30; two other-state roofs score 25 and 35. Of the
3 × 2 = 6 pairs, the other-state roof scores higher in 4 (25 beats 10 and 20; 35 beats all three), so AUC = 4/6
≈ 0.67.

In [ ]:
test_errors = metrics.reconstruction_errors(ae, te.images)          # one unusual score per test roof
variety_errors = metrics.reconstruction_errors(ae, variety.images)  # and per roof from the other states
print(f"AUC, other states vs Maine: {metrics.unusual_score_auc(test_errors, variety_errors):.2f}")

### Exercise 2 · The strangest roofs: an inspection queue (3 minutes)

The next cell shows the 10 test roofs with the **highest** unusual scores (the inspection queue) and the 10 with
the **lowest**. Look at both rows and discuss with your neighbour:
- What do the strangest roofs have in common? Would *you* send an inspector?
- Look at the most ordinary end. Is there a roof there that is not a house at all? Why would an empty lot or a
  patch of woods be *easy* to copy, and what does that say about what this score can miss?

In [ ]:
# Exercise 2: the inspection queue
r2 = exercises.strangest_roofs(ae, te.images, te.manifest, k=10, unusual_images=variety.images)

### Instructor demo · From one address to one score (3 minutes)

How little does it take to turn an address into a score about one property? The demo uses one **public
building**, South Portland City Hall, and asks two free geocoders where it is. Watch the screen: which square is
the building, and who decided which point to trust?

In [ ]:
# Instructor demo (you can run it afterwards). It uses a recorded OpenStreetMap answer, so it sends no request
# to OpenStreetMap; the instructor runs it once live in class with osm="live".
demo = pipeline.run_demo(ae, test_errors)

---
# Part 2 · ROOF-302 · Variational autoencoder (VAE)

> **Ticket ROOF-302.** *The claims system needs realistic test roofs that belong to nobody, and the team wants to
> see what lies "between" two roofs.*

A plain autoencoder's 128 numbers are scattered: pick 128 random numbers and the decoder draws nonsense. A **VAE**
makes that space **smooth**, so that any point near the middle decodes to a believable roof.

**The trick: the encoder gives a range, not a point.** For each of the 128 numbers it gives a **mean** $\mu$ and a
**spread** $\sigma$, and the code is drawn from them:

$$z = \mu + \sigma \cdot \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, 1)$$

| Symbol | Meaning |
|---|---|
| $\mu$ ("mu") | the centre of the range for one latent number |
| $\sigma$ ("sigma") | how wide the range is |
| $\varepsilon$ ("epsilon") | a random number from the standard bell curve (mean 0, spread 1) |
| $z$ | the latent number the decoder actually receives |

Writing the random draw this way (the *reparameterisation trick*) lets training adjust $\mu$ and $\sigma$.
**Worked example:** $\mu = 0.5$, $\sigma = 0.8$ and a draw $\varepsilon = -1.2$ give $z = 0.5 + 0.8 \times (-1.2) = -0.46$.

**The loss: copy well, and keep the ranges near a standard bell curve**

$$L = \frac{1}{B}\sum_{i=1}^{B}\Big[\lVert x_i - \hat{x}_i\rVert^2 \;+\; \beta\cdot \text{KL}_i\Big],
\qquad \text{KL} = -\tfrac{1}{2}\sum_{j=1}^{128}\big(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\big)$$

| Symbol | Meaning |
|---|---|
| $\lVert x_i - \hat{x}_i\rVert^2$ | the copy error, exactly as in Part 1 |
| KL | the *Kullback–Leibler divergence*: how far one roof's ranges are from a standard bell curve (mean 0, spread 1); never negative |
| $j$ | which of the 128 latent numbers |
| $\beta$ ("beta") | how much the KL term counts; 1 here |

**Worked example (one latent number).** With $\mu = 0.5$ and $\sigma = 0.8$: $\sigma^2 = 0.64$,
$\log 0.64 \approx -0.446$, so $1 - 0.446 - 0.25 - 0.64 = -0.336$ and KL $= -\tfrac12 \times (-0.336) = 0.168$.
With $\mu = 0$ and $\sigma = 1$: $1 + 0 - 0 - 1 = 0$, so KL $= 0$: no cost for matching the bell curve. The KL
term pulls every roof's range toward the middle, and that overlap is what makes the space smooth.

In [ ]:
# Train the VAE for 1 minute
set_seed(7)
if READY_MADE:
    vae = train.load_pretrained("vae")
else:
    vae = RoofVAE()
    h_vae = train.train_vae(vae, tr.images, watch=te.images[:8])
    viz.show_snapshots(h_vae)
    viz.plot_history(h_vae)

In [ ]:
# Brand-new roofs: decode 16 random points from the bell curve. These roofs belong to nobody.
with torch.no_grad():
    new_roofs = vae.sample(16, seed=1)
viz.show_grid(new_roofs, ncol=8, title="16 new roofs from the VAE: random points in its smooth roof space")
viz.latent_walk(vae, roof=te.images[2])     # move the 2 numbers (of 128) that change this roof most

### Exercise 3 · Morph one roof into another (3 minutes)

Pick two test roofs (any numbers from 0 to 999) and put them in `a` and `b` below. The VAE finds each roof's point
in its space and decodes the points on the straight line between them. Try two very different roofs (a dark roof
and a light one, a house and an empty lot).
*Do the middle pictures look like real roofs? Where does the change happen: smoothly, or all at once?*

In [ ]:
# Exercise 3: change a and b to any two test roofs from 0 to 999
r3 = exercises.morph_pair(vae, te.images, a=12, b=345, ae=ae)

---
# Part 3 · ROOF-303 · Generative adversarial network (GAN)

> **Ticket ROOF-303.** *The VAE's new roofs are blurry. The team wants sharper synthetic roofs, for training staff
> and for testing fraud detectors.*

A GAN has two networks that train **against each other**:
- the **generator** $G$ turns 100 random numbers $z$ into a roof $G(z)$;
- the **critic** (or *discriminator*) $D$ looks at a roof and gives the probability that it is real.

The critic learns to catch fakes; the generator learns to fool the critic. Nobody ever compares pixels with an
original, so nothing pushes the GAN toward a safe, blurry average: its roofs come out sharper.

**The two losses** (for one roof, using the *binary cross-entropy*, BCE):

$$L_D = -\big[\,0.9\,\log D(x) + 0.1\,\log\big(1 - D(x)\big)\big] \;-\; \log\big(1 - D(G(z))\big)
\qquad\qquad L_G = -\log D(G(z))$$

| Symbol | Meaning |
|---|---|
| $x$ | a real roof from the training set |
| $z$ | 100 random numbers, the generator's input |
| $G(z)$ | the generator's fake roof |
| $D(\cdot)$ | the critic's probability, between 0 and 1, that a roof is real |
| $\log$ | the natural logarithm: $\log 1 = 0$, and it heads to $-\infty$ as its input heads to 0, so a confident wrong answer costs a lot |
| 0.9 and 0.1 | *label smoothing*: the critic aims for 0.9 on real roofs instead of 1, so it never becomes over-confident |
| $L_D$ | the critic's loss: say ~0.9 for real roofs and 0 for fakes |
| $L_G$ | the generator's loss: make the critic say 1 for its fakes |

(Each loss is averaged over a batch of 128 roofs; the two networks take turns, one step each.)

**Worked example.** The critic gives a real roof $D(x) = 0.8$ and a fake $D(G(z)) = 0.3$.
- Critic, real roof: $-[0.9 \log 0.8 + 0.1 \log 0.2] = -[0.9(-0.223) + 0.1(-1.609)] \approx 0.362$.
- Critic, fake roof: $-\log(1 - 0.3) = -\log 0.7 \approx 0.357$. So $L_D \approx 0.72$.
- Generator: $-\log 0.3 \approx 1.20$. If the generator improves until $D(G(z)) = 0.6$, its loss drops to
  $-\log 0.6 \approx 0.51$, and the critic's loss on that fake rises to $-\log 0.4 \approx 0.92$. One's gain is the
  other's loss: that is the competition.

In [ ]:
# Train the GAN for 6 minutes (about 47 passes on a T4). While it trains, the instructor explains the pictures.
set_seed(7)
if READY_MADE:
    gen = train.load_pretrained("gan")
else:
    gen, critic = Generator(), Discriminator()
    h_gan = train.train_gan(gen, critic, tr.images)
    viz.show_snapshots(h_gan)     # the same 16 noise vectors, drawn as training went on
    viz.plot_history(h_gan)       # both losses, and the critic's average D(real) and D(fake)

In [ ]:
# 16 new roofs from the GAN, next to 16 real ones
with torch.no_grad():
    gan_roofs = data.to_bytes(gen.sample(16, seed=3)).cpu()
viz.show_rows({"GAN": gan_roofs[:8], "GAN ": gan_roofs[8:], "real": te.images[100:108]},
              title="New roofs from the GAN (top two rows) and real test roofs (bottom row)")
print(f"near-duplicates among 256 GAN roofs: "
      f"{metrics.near_duplicates(data.to_bytes(gen.sample(256, seed=1)).cpu(), te.images):.0%} (real roofs: 5%)")

**Near-duplicates** is the share of pairs of GAN roofs that are closer to each other than the closest 5% of pairs
of real roofs. Real roofs score 5% by construction. A GAN that keeps drawing the same roof (*mode collapse*)
scores far higher.

### Exercise 4 · Break the GAN on purpose (3 minutes)

Choose one setting below and run the cell. It trains a fresh GAN for 90 seconds with that one change.
- `"one_noise_number"`: the generator gets **1** random number instead of 100.
- `"critic_lr_x50"`: the critic learns **50 times faster** than the generator.
- `"none"`: the normal settings, to compare.

*Look at the 16 roofs and the near-duplicates number. What went wrong, and why would that change cause it?*
Compare with your neighbours, who may have picked another setting.

In [ ]:
# Exercise 4: pick "one_noise_number", "critic_lr_x50" or "none"
r4 = exercises.break_the_gan(tr.images, change="one_noise_number", minutes=1.5, real_images=te.images)

### Exercise 5 · Real or fake? (2 minutes)

The next cell shows 10 numbered roofs: some real, some from your GAN. Write down which numbers you think are
fake **before** running the cell after it, which reveals the answers.

In [ ]:
# Exercise 5: the quiz (write down your guesses)
quiz = exercises.real_or_fake(te.images, gen, n=10)

In [ ]:
# Exercise 5: the answers
quiz.reveal()

### Optional · The explorer

A small page with three tabs: **Inspect** (a test roof, its copy and its unusual score), **Morph** (any two of six
roofs through the VAE) and **New roof** (GAN roofs). Its pictures are prepared once, so it answers instantly.

In [ ]:
from genroofs import app
app.launch(ae, vae, gen, te.images, te.manifest, reference_errors=test_errors)

---
# Part 4 · Modern fixes: why our pictures blur, and what today's open models do

**Why the autoencoder and the VAE blur.** Two reasons:
1. **The squeeze.** 12,288 numbers go through 128. Big shapes survive; shingle lines and exact edges don't fit.
2. **The pixel-by-pixel score rewards smudging.** Take one pixel on the edge of a roof. From its 128 numbers the
   decoder can't tell whether that pixel is dark roof (−1) or bright lawn (+1); in similar training roofs it is each
   about half the time. Compare the average squared error of three guesses:

| The decoder's guess | Error if the pixel is −1 | Error if it is +1 | Average |
|---|---|---|---|
| dark, −1 | 0 | 4 | **2** |
| bright, +1 | 4 | 0 | **2** |
| grey, 0 | 1 | 1 | **1**, half the penalty |

So wherever it is unsure, grey is safest, and grey at every edge is blur. The VAE adds a little fuzz on purpose
(its ranges), so it blurs even more.

**Why the GAN is sharp but strange.** No pixel-by-pixel score, so no reward for grey: its edges are crisp. But in
6 minutes on 9,000 roofs it hasn't learned what a house *is*, so it draws crisp shapes that don't quite make sense.

**What later models changed.** Bigger memories that keep *where* things are (a small grid instead of one list),
scores for "does it look real" alongside "is it the same pixel", and millions of photos. Two free, open models
show one piece of this: they **enlarge a picture 4 times with believable detail**.
- **Real-ESRGAN** (Wang et al., 2021): a GAN, trained with a critic like ours. BSD-3-Clause.
- **SwinIR** (Liang et al., 2021): a Swin Transformer, trained the same way. Apache-2.0.

The next cell downloads both (about 67 MB each) and checks each file against its recorded fingerprint.

In [ ]:
# Load the two enlargers (about 20 seconds the first time)
esrgan = modern.load_upscaler("realesrgan")
swinir = modern.load_upscaler("swinir")

In [ ]:
# Four test roofs: the 64-pixel class photo, simply stretched, then enlarged by each model,
# with the sharper 128-pixel original the 64-pixel photo was made from ("the truth")
pick = [10, 120, 540, 660]
photos = te.images[pick]
truth = data.load_selected([te.manifest[i] for i in pick])     # the same roofs at 128 pixels, straight from the pack
modern.show_big({"the truth: 128-pixel original": truth,
                 "64-pixel photo, simple stretching": modern.stretch(photos),
                 "Real-ESRGAN": modern.enlarge(esrgan, photos),
                 "SwinIR": modern.enlarge(swinir, photos)},
                title="Enlarging the class photos 4 times (64 → 256 pixels)",
                note="Every picture is shown at 256 x 256, one screen pixel per picture pixel.")

In [ ]:
# Now our own models' pictures: can an enlarger fix the autoencoder's blur, or the GAN's strange shapes?
with torch.no_grad():
    ae_copies = exercises.copies(ae, photos)
modern.show_big({"autoencoder's copy, stretched": modern.stretch(ae_copies),
                 "Real-ESRGAN from the copy": modern.enlarge(esrgan, ae_copies),
                 "SwinIR from the copy": modern.enlarge(swinir, ae_copies),
                 "GAN roofs, stretched": modern.stretch(gan_roofs[:4]),
                 "Real-ESRGAN from the GAN roofs": modern.enlarge(esrgan, gan_roofs[:4]),
                 "SwinIR from the GAN roofs": modern.enlarge(swinir, gan_roofs[:4])},
                title="Enlarging our models' pictures")

**Sharp is not the same as true.** The cell below measures 64 test roofs against their 128-pixel originals:
*closeness* is PSNR (higher = closer to the truth) and *sharpness* is the average change between neighbouring
pixels, divided by the truth's (1.0 = as sharp as the original).

In [ ]:
rows = [te.manifest[i] for i in range(64)]
truth64 = data.load_selected(rows)                                  # 64 roofs at 128 pixels
small = te.images[:64]
at128 = lambda big: F.avg_pool2d(big.float(), 2).round().to(torch.uint8)   # 256 -> 128 by averaging 2 x 2 blocks
for name, big in [("simple stretching", modern.stretch(small)), ("Real-ESRGAN", modern.enlarge(esrgan, small)),
                  ("SwinIR", modern.enlarge(swinir, small))]:
    b = at128(big)
    print(f"{name:18s} closeness {metrics.psnr(b, truth64).mean():5.1f} dB   "
          f"sharpness {metrics.sharpness(b) / metrics.sharpness(truth64):.2f} of the truth")

**What to take away**
- The enlargers are **sharper** than stretching, but not **closer** to the truth: they draw *believable* detail,
  not the real detail.
- An enlarger can only sharpen detail that is **still in the picture**. Our autoencoder threw the detail away in
  the squeeze, so its enlarged copy is still a blurry copy. The fix for blur is a better model, not a bigger
  picture: the next cells run one.
- The slides show the Stable Diffusion family doing the same at a larger scale: very sharp pictures, often of a
  different house.

**A better autoencoder: Stable Diffusion's own.** Stable Diffusion draws its pictures inside the compressed space
of its own autoencoder (`sd-vae-ft-mse`, MIT licence, free to use). It is the same idea as ours, an encoder, a
middle and a decoder, with the three changes from the start of this part:
1. **A bigger memory that keeps *where* things are.** A 128-pixel roof becomes a grid of 4 × 16 × 16 = **1,024
   numbers**, one small set for every 8 × 8 patch. Ours squeezes a whole 64-pixel roof into one list of **128**.
2. **Better scores.** Besides "is it the same pixel", it was scored on "does it look the same to a network that
   recognises pictures" and by a critic asking "does it look real", the GAN idea.
3. **Millions of photos** instead of 9,000 roofs.

The next cell downloads it (about 335 MB, roughly 20 seconds, once per session) from one fixed version on Hugging
Face, as a `.safetensors` file that holds only numbers, and copies six test roofs at 128 pixels.

In [ ]:
# A better autoencoder: Stable Diffusion's own (about 335 MB the first time; MIT licence)
sd_ae = modern.load_sd_autoencoder()
pick6 = [10, 120, 540, 660, 12, 345]
photo128 = data.load_selected([te.manifest[i] for i in pick6])      # the 128-pixel originals
sd6 = modern.sd_copies(sd_ae, photo128)
with torch.no_grad():
    ours6 = exercises.copies(ae, te.images[pick6])
modern.show_big({"the photo (128 pixels)": photo128,
                 "our autoencoder's copy (64 pixels)": ours6,
                 "Stable Diffusion's autoencoder copy": sd6},
                px=192, title="The same roofs, copied by our autoencoder and by Stable Diffusion's",
                note="Shown at 192 x 192. Our copy is made at 64 pixels, so it is stretched to fit.")
print(f"Our autoencoder:                {metrics.psnr(ours6, te.images[pick6]).mean():5.1f} dB, "
      f"from 128 numbers for a 64-pixel roof")
print(f"Stable Diffusion's autoencoder: {metrics.psnr(sd6, photo128).mean():5.1f} dB, "
      f"from {modern.latent_numbers(128):,} numbers for a 128-pixel roof")

**Look closely.** The copies are sharp and faithful, but not perfect: fine lines, cars and small shadows can still
change. And sharper is not better for every job: in our tests this autoencoder was *worse* at flagging unusual
roofs, because it copies everything well, so nothing stands out. Detection and generation want different things.

---
# Part 5 · ROOF-304 · Ethics

> **Ticket ROOF-304.** *Before any of this goes near a customer: what could go wrong, and for whom?*

Four short checks, each with a question to discuss.

In [ ]:
# 1. Copying. Did the GAN reproduce a real home from its training set? Its 4 roofs closest to a training roof:
with torch.no_grad():
    fakes = data.to_bytes(gen.sample(64, seed=11)).cpu()
positions, distances = metrics.nearest_real(fakes, tr.images)
closest = distances.argsort()[:4]
viz.show_rows({"GAN roof": fakes[closest], "closest training roof": tr.images[positions[closest]]},
              title="The GAN's roofs that are closest to a real training roof")
print("distances:", [round(float(d), 3) for d in distances[closest]], "(0 would be an exact copy)")

**Discuss:** if a generator *did* reproduce a real home, whose home is it, and did its owner agree to be in a
training set? What changes when the training photos come from customers' claims?

In [ ]:
# 2. Fairness. Suppose the insurer sends an inspector to every roof above the 95th percentile of Maine scores.
#    What share of homes in each other state would be flagged?
cut = torch.quantile(test_errors.float(), 0.95)
print(f"Maine test roofs flagged: {(test_errors > cut).float().mean():.0%}")
for region in sorted({r["region"] for r in variety.manifest}):
    idx = torch.tensor([i for i, r in enumerate(variety.manifest) if r["region"] == region])
    print(f"{region:12s} flagged: {(variety_errors[idx] > cut).float().mean():.0%}")

**Discuss:** a model trained only on Maine roofs calls many homes elsewhere "unusual" simply because they look
different (tile roofs, flat roofs, pools, desert yards). If that score triggered inspections, or higher prices,
which homeowners would bear the cost? What data, checks and human review would you require first?

**3. One score about one home.** The demo in Part 1 turned an address into a score in seconds, and the two
geocoders didn't even agree which square was the building. *Who should be able to see that score, and could the
owner challenge it?*

**4. Fake evidence.** Your GAN, and far better models, can draw roofs that never existed, and enlargers add detail
that was never photographed. *How should an insurer treat an "AI-enhanced" or possibly generated claim photo? What
would you check: where the photo came from, the public aerial photo of the same house from before the storm, or
both?*

---
# Wrap-up

| You built | What it taught |
|---|---|
| An autoencoder | Squeezing forces a model to learn what is typical; a bad copy flags the unusual, and blur comes from the squeeze and the pixel score |
| A VAE | Ranges plus a KL term make a smooth space you can sample and walk through |
| A GAN | Competition instead of a pixel score gives sharp pictures, and new ways to fail (mode collapse) |
| Modern enlargers | Sharper is not truer; detail that was thrown away can only be invented |
| Ethics checks | Copying, fairness across places, one score per home, and fake evidence |

**Keep exploring:** everything here runs on your own laptop too. The setup guide is
[docs/SETUP.md](https://github.com/Sushanth-Manakhari/generative-models-aerial-imagery/blob/main/docs/SETUP.md), and the sources and licences are in
[docs/DATA_SOURCES.md](https://github.com/Sushanth-Manakhari/generative-models-aerial-imagery/blob/main/docs/DATA_SOURCES.md).